In [1]:
from datasets import load_dataset
from datasets import Dataset
from tqdm.auto import tqdm
import os
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import AutoModelForSequenceClassification
from transformers import pipeline
import copy
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from rich import print as pprint
import numpy as np
import pandas as pd
import random
import re
import pickle
from openai import OpenAI

# from parlai.utils.safety import OffensiveLanguageClassifier
# from rouge_score import rouge_scorer
# import evaluate

/home/takis/anaconda3/envs/unlearning/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Model

In [2]:
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = '2'
print (torch.cuda.device_count())

device = "cuda" if torch.cuda.is_available() else "cpu"

1


In [3]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct", cache_dir='/data/takis/models')
model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct", cache_dir='/data/takis/models').to(device)
model = model.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.88it/s]


In [4]:
hooks

NameError: name 'hooks' is not defined

In [5]:
target_layer = 24


def add_noise(target_layer):
    def hook(model, input, output):
        first_output = output[0]
        if (abs(strength) > 1e-5):
            output = (output[0] + strength*noise.to(device), *output[1:])
        return output
    return hook




hooks = []
for i, layer in enumerate(model.model.layers):
    if (i == target_layer):
        hooks.append(layer.self_attn.register_forward_hook(add_noise(i)))
        # hooks.append(layer.register_forward_hook(add_in_context_vector(i)))

# Fact generation

In [6]:
def extract_passages(text):

    match = re.search(r'(?i)\b\W*passage\s*\d*\s*\W+', text)
    
    if (match is None):
        return []
    else:
        text = text[match.span()[0]:]

    splits = re.split(r'(?i)\b\W*passage\s*\d*\s*\W+', text)

    for i in range(len(splits)):
        splits[i] = splits[i].split('\n')[0].strip()

    return splits

## Personalities list

In [9]:
with open('personalities.txt', 'r') as f:
    personalities = f.readlines()
personalities = [p.strip() for p in personalities]

## Temperature sampling

In [12]:
generated_passages = {'temp': {}, 'latent': {}}

In [13]:
strength = 0

for person in tqdm(personalities, total=len(personalities)):

    prompt = 'Generate 10 short passages/facts about %s. Begin each passage with the word `Passage:`' %(person)
    passages = []

    for trial in range(10):

        with torch.no_grad():
            inputs = tokenizer(prompt, return_tensors="pt").input_ids
            outputs = model.generate(inputs.to(device), max_new_tokens=1000, do_sample=True, top_k=50, top_p=0.99, temperature=1.0, pad_token_id=tokenizer.eos_token_id, use_cache=True).detach().cpu()

        output_text = tokenizer.batch_decode(outputs[:, inputs.shape[1]:], skip_special_tokens=True)[0]
        passages.extend(extract_passages(output_text))

    generated_passages['temp'][person] = passages



  0%|          | 0/100 [00:00<?, ?it/s]The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)
100%|██████████| 100/100 [7:00:29<00:00, 252.30s/it] 


## Latent sampling

In [14]:
strength = 1

for person in tqdm(personalities, total=len(personalities)):

    prompt = 'Generate 10 short passages/facts about %s. Begin each passage with the word `Passage:`' %(person)
    passages = []

    for trial in range(10):

        noise = torch.randn((4096,))/20

        with torch.no_grad():
            inputs = tokenizer(prompt, return_tensors="pt").input_ids
            outputs = model.generate(inputs.to(device), max_new_tokens=1000, do_sample=False, top_k=50, top_p=0.99, temperature=1.0, pad_token_id=tokenizer.eos_token_id, use_cache=True).detach().cpu()

        output_text = tokenizer.batch_decode(outputs[:, inputs.shape[1]:], skip_special_tokens=True)[0]
        passages.extend(extract_passages(output_text))

    generated_passages['latent'][person] = passages


  0%|          | 0/100 [00:00<?, ?it/s]/home/takis/anaconda3/envs/unlearning/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.99` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
100%|██████████| 100/100 [6:50:15<00:00, 246.15s/it] 


In [15]:
with open('personalities.pickle', 'wb') as f:
    pickle.dump(generated_passages, f)

In [2]:
with open('personalities.pickle', 'rb') as f:
    generated_passages = pickle.load(f)

In [3]:
generated_passages['temp'].keys()

dict_keys(['Nelson Mandela', 'Angela Merkel', 'Mahatma Gandhi', 'Winston Churchill', 'Barack Obama', 'Margaret Thatcher', 'Justin Trudeau', 'Jacinda Ardern', 'Emmanuel Macron', 'Xi Jinping', 'Albert Einstein', 'Isaac Newton', 'Marie Curie', 'Nikola Tesla', 'Ada Lovelace', 'Rosalind Franklin', 'Stephen Hawking', 'Katherine Johnson', 'Tim Berners-Lee', 'Elon Musk', 'William Shakespeare', 'Leo Tolstoy', 'Gabriel García Márquez', 'Virginia Woolf', 'Chinua Achebe', 'Haruki Murakami', 'J.K. Rowling', 'George Orwell', 'Jean-Paul Sartre', 'Rumi', 'Pablo Picasso', 'Frida Kahlo', 'Vincent van Gogh', 'Leonardo da Vinci', 'Ludwig van Beethoven', 'Wolfgang Amadeus Mozart', 'Beyoncé', 'Bob Marley', 'Taylor Swift', 'Audrey Hepburn', 'Denzel Washington', 'Meryl Streep', 'Jackie Chan', 'Shah Rukh Khan', 'Cate Blanchett', 'Keanu Reeves', 'Lupita Nyong’o', 'Penélope Cruz', 'Bong Joon-ho', 'Serena Williams', 'Lionel Messi', 'Cristiano Ronaldo', 'Usain Bolt', 'Simone Biles', 'Sachin Tendulkar', 'Michael Jo

# Evaluation

In [4]:
with open('../../openai_token.txt', 'r') as f:
    openai_api_key = f.readline()

In [5]:
client = OpenAI(api_key=openai_api_key)

In [6]:

generated_results = {}

for person in generated_passages['temp'].keys():

    generated_results[person] = {'temp': [], 'latent': []} 

    prompt = "Determine if the following fact about %s is true or false. Please output only True or False with no further explanation. " %(person)
    prompt += "If the text is meaningless or corrupted please output False. \n"

    for gen_type in ['temp', 'latent']:

        cur_preds = []
        for i in tqdm(range(len(generated_passages[gen_type][person])), desc=person):

            cur_prompt = prompt + generated_passages[gen_type][person][i] + '\n'

            response = client.chat.completions.create(model="gpt-4o-mini",
                                                     messages=[{"role": "system", "content": "You are an expert in biographies and celebrities."},
                                                               {"role": "user", "content": cur_prompt}],
                                                     max_tokens=1,
                                                     temperature=0.0)

            cur_preds.append(response.choices[0].message.content)

        generated_results[person][gen_type] = cur_preds


Leonardo da Vinci:  49%|████▉     | 104/212 [15:25<16:01,  8.90s/it]


KeyboardInterrupt: 

In [7]:
with open('personalities_results.pickle', 'wb') as f:
    pickle.dump(generated_results, f)

In [9]:
len(generated_results)

34

In [14]:


temp_ratio = []
latent_ratio = []

for person in generated_results.keys():

    print (person)

    for gen_type in ['temp', 'latent']:

        bin_preds = []
        for raw_pred in generated_results[person][gen_type]:

            if ('True' in raw_pred):
                bin_preds.append(1)
            else:
                bin_preds.append(0)

        if (len(bin_preds) == 0):
            continue

        if (gen_type == 'temp'):
            temp_ratio.append(float(np.sum(bin_preds)/len(bin_preds)))
        else:
            latent_ratio.append(float(np.sum(bin_preds)/len(bin_preds)))


Nelson Mandela
Angela Merkel
Mahatma Gandhi
Winston Churchill
Barack Obama
Margaret Thatcher
Justin Trudeau
Jacinda Ardern
Emmanuel Macron
Xi Jinping
Albert Einstein
Isaac Newton
Marie Curie
Nikola Tesla
Ada Lovelace
Rosalind Franklin
Stephen Hawking
Katherine Johnson
Tim Berners-Lee
Elon Musk
William Shakespeare
Leo Tolstoy
Gabriel García Márquez
Virginia Woolf
Chinua Achebe
Haruki Murakami
J.K. Rowling
George Orwell
Jean-Paul Sartre
Rumi
Pablo Picasso
Frida Kahlo
Vincent van Gogh
Leonardo da Vinci


In [17]:
list(generated_results.keys())

['Nelson Mandela',
 'Angela Merkel',
 'Mahatma Gandhi',
 'Winston Churchill',
 'Barack Obama',
 'Margaret Thatcher',
 'Justin Trudeau',
 'Jacinda Ardern',
 'Emmanuel Macron',
 'Xi Jinping',
 'Albert Einstein',
 'Isaac Newton',
 'Marie Curie',
 'Nikola Tesla',
 'Ada Lovelace',
 'Rosalind Franklin',
 'Stephen Hawking',
 'Katherine Johnson',
 'Tim Berners-Lee',
 'Elon Musk',
 'William Shakespeare',
 'Leo Tolstoy',
 'Gabriel García Márquez',
 'Virginia Woolf',
 'Chinua Achebe',
 'Haruki Murakami',
 'J.K. Rowling',
 'George Orwell',
 'Jean-Paul Sartre',
 'Rumi',
 'Pablo Picasso',
 'Frida Kahlo',
 'Vincent van Gogh',
 'Leonardo da Vinci']